# qPISA EV1/EV4: endpoint measurements and MEROPS-position windows

Read both deposited endpoint experiments and associate every possible measurement with its substrate window. Keep every signed change, including zero and negative values; no cleavage threshold is imposed.

## Definitions

Inputs: `MOESM2` (EV1: DPP4 and DPF-3) and `MOESM5` (EV4: separate DPF-3 repeat), under `raw/dpp4_qpisa`. Publication: DOI `10.1038/s44320-024-00071-4`. Original normalized log2 peptide levels and source row/column names are retained.

For `Tryptic+` input peptides, **substrate_log2_depletion = control − enzyme**. The DPP-IV-family cut follows residue two; the six observed residue positions are aligned as `--AB|CDEF`. `site_evidence=enzyme_geometry` distinguishes this convention from a source-reported peptide-specific cut.

For `Non-Tryptic+` products, **product_log2_enrichment = enzyme − control**. Recover a parent only when exactly one deposited input sequence equals two N-terminal residues followed by that product. Its evidence is `unique_sequence_match_in_deposit`, not a supplied experimental pairing. Zero or multiple candidates remain unresolved, without deleting the measurement. Different experiments and enzymes are never pooled.

`treated_to_control_signal_ratio = 2 ** (enzyme_log2_level − control_log2_level)`; `relative_signal_loss = 1 − ratio` is also retained for input peptides. These describe relative normalized signals, not calibrated molar amounts. Conditions from article methods: 4 h, 21 °C, pH 7.4; enzyme doses differ between DPP4 and DPF-3. No kcat is inferred and no intermediate files are written.

In [1]:
from pathlib import Path
import sys
import logging

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "assets/tools/proteolysis_data_manager").is_dir())
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
RAW = ROOT / "data/proteolysis/window_neighbourhood_datasets/raw"
pd.set_option("display.max_columns", None)
logging.getLogger("proteolysis_data_manager.analysis.window_neighbourhood.qpisa").setLevel(logging.WARNING)
logging.getLogger("proteolysis_data_manager.analysis.window_neighbourhood.pics").setLevel(logging.WARNING)

from proteolysis_data_manager.analysis.window_neighbourhood.qpisa import read_endpoint_measurements
measurements = read_endpoint_measurements(RAW / "dpp4_qpisa")
unresolved_measurements = measurements.loc[~measurements["has_window"]]

In [2]:
overview = measurements.groupby(["experiment", "enzyme", "measurement_type"], sort=False).agg(
    measurement_rows=("value", "size"), with_window=("has_window", "sum"),
    distinct_windows=("window", "nunique"), median_value=("value", "median"),
).reset_index()
display(overview.round(4))
window_evidence = measurements.groupby(["experiment", "enzyme", "site_evidence"], sort=False).size().rename("measurement_rows").reset_index()
display(window_evidence)

# Retain source association and positional invariants without dropping measurements.
assert measurements["source_record_id"].is_unique
assert np.isfinite(measurements["value"]).all()
assert measurements["P4"].isna().all() and measurements["P3"].isna().all()
resolved = measurements.loc[measurements["has_window"]]
assert resolved["window"].eq("--" + resolved["substrate_sequence"].str[:2] + "|" + resolved["substrate_sequence"].str[2:6]).all()
matched_products = measurements.loc[measurements["site_evidence"].eq("unique_sequence_match_in_deposit")]
assert matched_products["substrate_sequence"].str[2:].eq(matched_products["measured_sequence"]).all()
assert unresolved_measurements["window"].isna().all()

,experiment,enzyme,measurement_type,measurement_rows,with_window,distinct_windows,median_value
0,EV1,DPF-3,substrate_log2_depletion,43314,43314,41725,-0.0020
1,EV1,DPF-3,product_log2_enrichment,10185,2580,2523,0.1159
2,EV1,DPP4,substrate_log2_depletion,43314,43314,41725,-0.0643
3,EV1,DPP4,product_log2_enrichment,10185,2580,2523,0.1613
4,EV4,DPF-3,substrate_log2_depletion,47430,47430,45324,0.0079
5,EV4,DPF-3,product_log2_enrichment,10748,2559,2490,0.1701


,experiment,enzyme,site_evidence,measurement_rows
0,EV1,DPF-3,enzyme_geometry,43314
1,EV1,DPF-3,unresolved_parent,7601
2,EV1,DPF-3,unique_sequence_match_in_deposit,2580
3,EV1,DPF-3,ambiguous_parent,4
4,EV1,DPP4,enzyme_geometry,43314
5,EV1,DPP4,unresolved_parent,7601
6,EV1,DPP4,unique_sequence_match_in_deposit,2580
7,EV1,DPP4,ambiguous_parent,4
8,EV4,DPF-3,enzyme_geometry,47430
9,EV4,DPF-3,unresolved_parent,8185


## Inspect the extracted records

The selection controls only the preview. `measurements` contains every original endpoint observation, and `unresolved_measurements` retains products whose parent context cannot be established. A positive value has the meaning given by its `measurement_type`; substrate depletion and product enrichment are not combined into one undifferentiated target.

In [3]:
EXPERIMENT = "EV1"
ENZYME = "DPP4"
PREVIEW_ROWS_PER_EVIDENCE = 2

selected_measurements = measurements.loc[measurements["experiment"].eq(EXPERIMENT) & measurements["enzyme"].eq(ENZYME)].copy()
columns = ["measured_sequence", "window", "site_evidence", "measurement_type", "value", "unit", "treated_to_control_signal_ratio", "source_excel_row"]
display(selected_measurements.groupby("site_evidence", sort=False).head(PREVIEW_ROWS_PER_EVIDENCE)[columns].reset_index(drop=True))

,measured_sequence,window,site_evidence,measurement_type,value,unit,treated_to_control_signal_ratio,source_excel_row
0,AAAAAAAAAAAAAAAGAGAGAK,--AA|AAAA,enzyme_geometry,substrate_log2_depletion,0.656903,log2_fold_change,0.634238,2
1,AAAAAAAAAAGAAGGR,<NA>,unresolved_parent,product_log2_enrichment,-0.095592,log2_fold_change,0.935888,3
2,AAAAAAAAAPAAAATAPTTAATTAATAAQ,--AA|AAAA,enzyme_geometry,substrate_log2_depletion,0.120282,log2_fold_change,0.920008,4
3,AAAAAAAGDSDSWDADAFSVEDPVRK,<NA>,unresolved_parent,product_log2_enrichment,0.086195,log2_fold_change,1.061567,5
4,AAAGIFLNNVK,--QA|AAAG,unique_sequence_match_in_deposit,product_log2_enrichment,-0.196113,log2_fold_change,0.872899,26
5,AAAGYDVEK,--AL|AAAG,unique_sequence_match_in_deposit,product_log2_enrichment,-0.715416,log2_fold_change,0.609030,29
6,AGDTHLGGEDFDNR,<NA>,ambiguous_parent,product_log2_enrichment,5.075225,log2_fold_change,33.712809,1266
7,FDDAIAELDTLNEDSYK,<NA>,ambiguous_parent,product_log2_enrichment,3.024757,log2_fold_change,8.138468,14606


### Remarks

These are endpoint measurements paired with window coordinates, not unqualified counts of observed cleavage events. Product sequence matches remain a separate evidence class. DPF-3 has no invented MEROPS accession. The normalized signal ratio can exceed one and its corresponding input loss can be negative; both are retained.

### Notes

## Results / Summary

### LLM

The extraction retains the complete EV1 and EV4 measurements with their original levels, signed response, conditions, source provenance and available window. `has_window` and `site_evidence` separate usable contexts from unresolved parents without excluding observations.

### Person

## Common numerical parameters

The common columns are remaining percent = 100 × 2⁻ᵈ and depletion percent = 100 × (1 − 2⁻ᵈ), where d is the reported log₂ control/treated difference. Negative net loss remains negative. The apparent exponential rate is ln(2) × d/time. These describe control-relative signals. Endpoint candidate-window frequencies use positive fractional depletion as a continuous weight under the approximately mixed input assumption; they are not thresholded cleavage counts. All transformations stay in memory; no CSV or database is written.

In [4]:
# Use the independent data library from this checkout.
import sys
sys.path.insert(0, str(ROOT / "assets/tools/proteolysis_data_manager/src"))
from proteolysis_data_manager.integration.merge import (
    build_dataset, parse_dataset, add_merops_scores,
)
contract = build_dataset(
    ROOT / "data/proteolysis", sources=["qpisa_ev1", "qpisa_ev4"],
)
contract = add_merops_scores(
    contract, ROOT / "data/proteolysis/window_neighbourhood_datasets/raw/merops",
    dataset_index=34,
)
common_parameters = contract.to_frame()
preview = common_parameters.loc[common_parameters.substrate_depletion_percent.notna()]
display(preview.groupby(["source_dataset", "enzyme_name"], dropna=False, sort=False).head(2)[[
    "source_db", "source_dataset", "enzyme_name", "window", "time_s",
    "substrate_remaining_percent", "substrate_depletion_percent", "apparent_rate_s_inv",
    "position_probability_sum", "probability_positions_scored",
]])


,source_db,source_dataset,enzyme_name,window,time_s,substrate_remaining_percent,substrate_depletion_percent,apparent_rate_s_inv,position_probability_sum,probability_positions_scored
0,qpisa,EV1,DPF-3,- - A A | A A A A,14400,98.494705,1.505295,1.053291e-06,0.667888,6
2,qpisa,EV1,DPF-3,- - A A | A A A A,14400,90.841427,9.158573,6.670469e-06,0.667888,6
53499,qpisa,EV1,DPP4,- - A A | A A A A,14400,63.423834,36.576166,3.162017e-05,0.852797,6
53501,qpisa,EV1,DPP4,- - A A | A A A A,14400,92.000771,7.999229,5.789808e-06,0.852797,6
106998,qpisa,EV4,DPF-3,- - A A | A A A A,14400,98.735707,1.264293,8.835788e-07,0.694611,6
106999,qpisa,EV4,DPF-3,- - A A | A A A A,14400,90.953967,9.046033,6.584490e-06,0.694611,6
